# Расширение обучения до 6000 запросов

Эксперимент изменяет объём обучения при тех же признаках и методах. Матрицы TF-IDF хранятся в CSC для исключения повторных преобразований при пакетном поиске. Индексы и обучающие пары сохраняются в outputs/v2.

Метод выбирается по прежней dev-выборке, с обязательным сравнением с принятой моделью. Финальная проверка использует 500 запросов из прежнего test-пула, исключая все тексты, просмотренные в старом test. Результат записывается в answer_v2.csv; принятый answer.csv не перезаписывается.

Запуск ограничен 80 минутами. Значения метрик появятся после выполнения; улучшение заранее не гарантируется.


## 1. Настройки

Фиксирован seed 42. Объём эксперимента ограничен 6000 обучающими группами, 500 группами dev и 500 группами test. Каждый канал поиска возвращает до 100 объявлений.

In [1]:
import json
import os
import re
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits

SEED = 42
TRAIN_SIZE = 6000
DEV_SIZE = 500
TEST_SIZE = 500
TOP_K = 50
CHANNEL_SIZE = 100
BATCH_SIZE = 32

os.makedirs("outputs", exist_ok=True)
start_time = time.time()
import joblib
previous_model = joblib.load("outputs/v1_backup/selector.joblib")["model"]


## 2. Данные

Из train загружаются условия запроса и идентификатор выбранного объявления. Признаки кандидатов берутся из benchmark_items, общего корпуса для обучения отбора и итогового поиска.

In [2]:
search_columns = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

train = pd.read_parquet("data/train.parquet", columns=search_columns + ["item_id"])
queries = pd.read_parquet("data/benchmark_queries.parquet")
items = pd.read_parquet("data/benchmark_items.parquet")

print("Строк обучения:", len(train))
print("Запросов бенчмарка:", len(queries))
print("Объявлений в корпусе:", len(items))

Строк обучения: 497673
Запросов бенчмарка: 2452
Объявлений в корпусе: 189212


In [3]:
train.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_id
0,скупка телевизоров,652430,0,,114,e8b685dffe1a408e
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,92e1b0446f827b59
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,624846856ce81d69


In [4]:
queries.head(3)

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114


In [5]:
items.head(3)

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114


## 3. Проверка данных

Проверяются идентификаторы, пропуски и пересечения train с корпусом. Идентификаторы обрабатываются как строки.

In [6]:
assert queries["query_id"].notna().all(), "Есть пустые query_id"
assert queries["query_id"].is_unique, "Повторяются query_id"
assert (queries["query_id"].str.len() == 16).all(), "Неверная длина query_id"
assert items["item_id"].is_unique, "Повторяются item_id"
assert items["item_id"].str.fullmatch("[0-9a-f]{16}", na=False).all(), "Неверный item_id"

print("Типы полей запроса:")
print(queries.dtypes)
print("\nПропуски в корпусе:")
print(items.isna().sum())

Типы полей запроса:
query_id                       str
search_query                   str
search_location_id           int64
search_is_delivery_search    int32
search_infm_params_text        str
search_category              int64
dtype: object

Пропуски в корпусе:
item_title_raw                   0
item_rating_reviews_count    11615
item_rating                  17631
item_price                       0
item_microcat_id                 0
item_longitude                   1
item_location_id                 0
item_latitude                    1
item_is_phone_hidden             0
item_is_message_forbidden        0
item_infm_params_text            0
item_id                          0
item_description_raw            35
item_category_id                 0
dtype: int64


In [7]:
print("Уникальных текстов в train:", train["search_query"].nunique())
print("Повторов условий в benchmark:", queries.duplicated(search_columns).sum())

known_queries = queries["search_query"].isin(train["search_query"])
known_items = train["item_id"].isin(items["item_id"])

print("Текст запроса был в train (%):", round(known_queries.mean() * 100, 2))
print("Строк train с объявлением из корпуса (%):", round(known_items.mean() * 100, 2))

Уникальных текстов в train: 74529
Повторов условий в benchmark: 0


Текст запроса был в train (%): 36.99
Строк train с объявлением из корпуса (%): 6.63


## 4. Подготовка текста

Используются нижний регистр, замена «ё» на «е», удаление пунктуации и повторяющихся пробелов. Для словарного индекса объединены два заголовка, параметры и первые 1500 символов описания. Повтор заголовка увеличивает его вес. Символьный индекс использует только заголовок и параметры для ограничения памяти.

In [8]:
def clean_text(text):
    """Очищает одну строку: убирает пунктуацию и лишние пробелы."""
    if pd.isna(text):
        return ""

    text = text.lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print(clean_text("  РЕМОНТ    телевизоров!!!  "))

ремонт телевизоров


In [9]:
items["clean_title"] = items["item_title_raw"].map(clean_text)
items["clean_parameters"] = items["item_infm_params_text"].map(clean_text)
descriptions = items["item_description_raw"].str.slice(0, 1500).map(clean_text)

# Заголовок повторён дважды, чтобы его слова сильнее влияли на поиск.
items["search_text"] = (
    items["clean_title"] + " " + items["clean_title"] + " "
    + items["clean_parameters"] + " " + descriptions
)

# Для частей слов берём короче: так индекс требует меньше памяти.
items["short_text"] = items["clean_title"] + " " + items["clean_parameters"]
queries["clean_query"] = queries["search_query"].map(clean_text)

del descriptions
items[["item_title_raw", "search_text"]].head(3)

,item_title_raw,search_text
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,ремонт выкуп компьют и ноутбуков с выездом на ...
1,Обучение ребенка чтению,обучение ребенка чтению обучение ребенка чтени...
2,Афрокудри 5+,афрокудри 5 афрокудри 5 вид услуги красота здо...


## 5. Запросы с доступной разметкой

Пары train объединяются по пяти условиям поиска. Повторные выборы одного item_id удаляются. Для локального эксперимента остаются только группы, все известные ответы которых присутствуют в корпусе: 18 436 из 354 463. Такой отбор смещает распределение локальной выборки относительно бенчмарка.

In [10]:
groups = train.groupby(search_columns, dropna=False)["item_id"].agg(set)
groups = groups.reset_index(name="relevant_items")

corpus_ids = set(items["item_id"])
can_evaluate = []

for relevant_items in groups["relevant_items"]:
    # Все известные ответы должны существовать в корпусе.
    all_answers_exist = relevant_items.issubset(corpus_ids)
    can_evaluate.append(all_answers_exist)

eligible = groups[can_evaluate].copy()
eligible["clean_query"] = eligible["search_query"].map(clean_text)
eligible = eligible.reset_index(drop=True)

print("Всего групп запросов:", len(groups))
print("Все известные ответы есть в корпусе:", len(eligible))
assert len(eligible) > 0, "Нет запросов для проверки"

Всего групп запросов: 354463
Все известные ответы есть в корпусе: 18436


## 6. Обучение, dev и test

Разделение проводится по очищенным текстам запросов: один текст не пересекается между частями, даже при разных локациях. Примерное соотношение уникальных текстов — 70/15/15; после разделения выбираются 6000/500/500 групп условий. На train обучается бустинг, на dev выбирается метод, test используется для итоговой оценки.

In [11]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_positions, other_positions = next(
    splitter.split(eligible, groups=eligible["clean_query"])
)
train_pool = eligible.iloc[train_positions]
other_pool = eligible.iloc[other_positions]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
dev_positions, test_positions = next(
    splitter.split(other_pool, groups=other_pool["clean_query"])
)
dev_pool = other_pool.iloc[dev_positions]
test_pool = other_pool.iloc[test_positions]

train_size = min(TRAIN_SIZE, len(train_pool))
train_part = train_pool.sample(n=train_size, random_state=SEED)
dev_size = min(DEV_SIZE, len(dev_pool))
dev_part = dev_pool.sample(n=dev_size, random_state=SEED)
# Контрольные тексты прежнего эксперимента повторно не используем.
old_test = test_pool.sample(n=min(500, len(test_pool)), random_state=SEED)
old_test_texts = set(old_test["clean_query"])
fresh_mask = ~test_pool["clean_query"].isin(old_test_texts)
test_pool = test_pool[fresh_mask]
assert len(test_pool) >= TEST_SIZE, "Недостаточно новых контрольных запросов"
test_size = min(TEST_SIZE, len(test_pool))
test_part = test_pool.sample(n=test_size, random_state=SEED)

train_part = train_part.reset_index(drop=True)
dev_part = dev_part.reset_index(drop=True)
test_part = test_part.reset_index(drop=True)

train_texts = set(train_part["clean_query"])
dev_texts = set(dev_part["clean_query"])
test_texts = set(test_part["clean_query"])
assert train_texts.isdisjoint(dev_texts)
assert train_texts.isdisjoint(test_texts)
assert dev_texts.isdisjoint(test_texts)

print("Обучение:", len(train_part))
print("Выбор метода:", len(dev_part))
print("Финальная проверка:", len(test_part))
assert set(test_part["clean_query"]).isdisjoint(old_test_texts)


Обучение: 6000
Выбор метода: 500
Финальная проверка: 500


## 7. Локация и категория

Доли совпадений рассчитаны только по обучающей части. Локация используется как мягкий сигнал: поиск не исключает объявления из других мест. Код категории 0 трактуется как отсутствие конкретного ограничения.

In [12]:
items_by_id = items.set_index("item_id")
same_locations = []
same_categories = []

for _, row in train_part.iterrows():
    for item_id in row["relevant_items"]:
        item = items_by_id.loc[item_id]
        same_locations.append(row["search_location_id"] == item["item_location_id"])
        if row["search_category"] != 0:
            same_categories.append(row["search_category"] == item["item_category_id"])

print("Совпадение локации у выбранных объявлений (%):", round(np.mean(same_locations) * 100, 2))
print("Совпадение указанной категории (%):", round(np.mean(same_categories) * 100, 2))

Совпадение локации у выбранных объявлений (%): 79.97
Совпадение указанной категории (%): 100.0


## 8. Текстовые индексы

Словарный TF-IDF: min_df=2, до 100 000 признаков, логарифмическая частота, float32. Нормировка L2 позволяет использовать скалярное произведение как косинусное сходство. Индекс строится по доступному корпусу, без меток dev и test.

In [13]:
word_vectorizer = TfidfVectorizer(
    min_df=2,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32,
)
word_vectors = word_vectorizer.fit_transform(items["search_text"]).tocsc()
print("Размер матрицы слов:", word_vectors.shape)
# CSC даёт CSR при .T: преобразование не повторяется для каждого пакета.


Размер матрицы слов: (189212, 100000)


### Символьный TF-IDF

Фрагменты длиной 3–5 символов, analyzer=char_wb, min_df=3, до 150 000 признаков. Метод позволяет учитывать близкие формы слов и некоторые опечатки, но не является полноценным семантическим поиском.

In [14]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=150000,
    sublinear_tf=True,
    dtype=np.float32,
)
char_vectors = char_vectorizer.fit_transform(items["short_text"]).tocsc()
print("Размер матрицы частей слов:", char_vectors.shape)
# Сохраняем индексы для возможного продолжения эксперимента.
joblib.dump({"word_vectorizer": word_vectorizer, "char_vectorizer": char_vectorizer,
             "word_vectors": word_vectors, "char_vectors": char_vectors},
            "outputs/v2/indexes.joblib")
# Освобождаем длинные тексты после построения индексов.
items_by_id = items[["item_id", "item_title_raw"]].set_index("item_id")
items = items.drop(columns=["item_description_raw", "search_text", "short_text"])


Размер матрицы частей слов: (189212, 150000)


## 9. Расширенный список кандидатов

Объединяются четыре списка: по словам и частям слов во всём корпусе, а также те же два списка внутри заданной локации. Каждый содержит до 100 положительных совпадений. После удаления повторов остаётся до 400 кандидатов. При пустом тексте используется текст фильтров. Равные оценки разрешаются стабильной сортировкой по исходному порядку.

In [15]:
item_ids = items["item_id"].to_numpy()
item_locations = items["item_location_id"].to_numpy()
item_categories = items["item_category_id"].to_numpy()

location_positions = items.groupby("item_location_id").indices


def top_positions(scores, limit):
    """Возвращает до limit позиций с наибольшими положительными оценками."""
    positive_mask = scores > 0
    positions = np.flatnonzero(positive_mask)
    positive_scores = scores[positions]

    order = np.argsort(-positive_scores, kind="stable")
    best_order = order[:limit]
    return positions[best_order]

### Пакетный расчёт

Запросы обрабатываются пакетами по 32 для ограничения памяти. Сохраняются только позиции кандидатов и их оценки; правильные ответы при поиске не используются.

In [16]:
def retrieve_candidates(table):
    """Находит до 400 кандидатов для каждого запроса из table."""
    records = []

    for start in range(0, len(table), BATCH_SIZE):
        batch = table.iloc[start:start + BATCH_SIZE]
        texts = []

        for _, row in batch.iterrows():
            text = clean_text(row["search_query"])
            if text == "":
                text = clean_text(row["search_infm_params_text"])
            texts.append(text)

        query_words = word_vectorizer.transform(texts)
        query_chars = char_vectorizer.transform(texts)
        word_scores = query_words @ word_vectors.T
        word_scores = word_scores.toarray()
        char_scores = query_chars @ char_vectors.T
        char_scores = char_scores.toarray()

        for number in range(len(batch)):
            row = batch.iloc[number]
            words = word_scores[number]
            chars = char_scores[number]

            word_best = top_positions(words, CHANNEL_SIZE)
            char_best = top_positions(chars, CHANNEL_SIZE)
            local = location_positions.get(row["search_location_id"], np.array([], dtype=int))
            local_word_best = local[top_positions(words[local], CHANNEL_SIZE)]
            local_char_best = local[top_positions(chars[local], CHANNEL_SIZE)]

            positions = np.unique(np.concatenate([
                word_best, char_best, local_word_best, local_char_best,
            ]))

            records.append({
                "positions": positions,
                "word_score": words[positions],
                "char_score": chars[positions],
                "baseline": word_best[:TOP_K],
            })

        if (start // BATCH_SIZE + 1) % 10 == 0:
            print("Обработано запросов:", min(start + BATCH_SIZE, len(table)))

    return records

## 10. Базовая оценка и полнота кандидатов

Базовый вариант — топ-50 словарного TF-IDF. Дополнительно измеряется Recall всего расширенного списка: это верхняя граница для последующего отбора из этого списка. Все методы сравниваются на одинаковых запросах.

In [17]:
train_records = retrieve_candidates(train_part)
dev_records = retrieve_candidates(dev_part)
joblib.dump({"train_part": train_part, "dev_part": dev_part, "test_part": test_part,
             "train_records": train_records, "dev_records": dev_records},
            "outputs/v2/candidates.joblib")


def recall_for_query(predicted_positions, relevant_items):
    """Считает долю найденных правильных ответов одного запроса."""
    predicted_ids = set(item_ids[predicted_positions])
    found = predicted_ids.intersection(relevant_items)
    return len(found) / len(relevant_items)


def evaluate_lists(table, predictions):
    """Возвращает средний Recall и массив оценок отдельных запросов."""
    recalls = []
    for relevant_items, predicted_positions in zip(table["relevant_items"], predictions):
        recalls.append(recall_for_query(predicted_positions, relevant_items))
    return float(np.mean(recalls)), np.array(recalls)


dev_baseline, _ = evaluate_lists(dev_part, [record["baseline"] for record in dev_records])
dev_ceiling, _ = evaluate_lists(dev_part, [record["positions"] for record in dev_records])
print("Базовый Recall@50 на dev:", round(dev_baseline, 4))
print("Recall расширенного списка на dev:", round(dev_ceiling, 4))

Обработано запросов: 320


Обработано запросов: 640


Обработано запросов: 960


Обработано запросов: 1280


Обработано запросов: 1600


Обработано запросов: 1920


... (10 outputs left out)

Обработано запросов: 5440


Обработано запросов: 5760


Обработано запросов: 320


Базовый Recall@50 на dev: 0.294
Recall расширенного списка на dev: 0.865


## 11. Отбор без обучения

Сумма двух текстовых сходств умножается на 1 + location_weight при совпадении локации. На dev сравниваются веса 0, 0.5, 1 и 2. Жёсткие фильтры категории и локации не применяются.

In [18]:
def hybrid_predictions(table, records, location_weight):
    """Выбирает до 50 объявлений по текстовому сходству и локации."""
    predictions = []

    for number in range(len(table)):
        row = table.iloc[number]
        record = records[number]
        positions = record["positions"]
        same_location = item_locations[positions] == row["search_location_id"]
        scores = record["word_score"] + record["char_score"]
        scores = scores * (1 + location_weight * same_location)
        order = np.argsort(-scores, kind="stable")[:TOP_K]
        predictions.append(positions[order])

    return predictions


results = [{"method": "word_tfidf", "dev_recall_at_50": dev_baseline}]
hybrid_weights = {}

for weight in [0.0, 0.5, 1.0, 2.0]:
    name = "hybrid_" + str(weight)
    hybrid_weights[name] = weight
    predictions = hybrid_predictions(dev_part, dev_records, weight)
    score, _ = evaluate_lists(dev_part, predictions)
    results.append({"method": name, "dev_recall_at_50": score})

pd.DataFrame(results)

,method,dev_recall_at_50
0,word_tfidf,0.294000
1,hybrid_0.0,0.293000
2,hybrid_0.5,0.549000
3,hybrid_1.0,0.690667
4,hybrid_2.0,0.756000


## 12. Признаки отбора

Используются 18 признаков: два текстовых сходства, их сумма и произведение; совпадения локации и категории; наличие категории; длина запроса; доставка; доля слов фильтров в параметрах; рейтинг и его пропуск; log(1 + число отзывов) и пропуск; log(1 + цена) и пропуск; доступность телефона и сообщений. Идентификаторы запросов и объявлений в модель не передаются.

In [19]:
item_ratings = pd.to_numeric(items["item_rating"], errors="coerce").to_numpy(dtype=float)
item_reviews = pd.to_numeric(items["item_rating_reviews_count"], errors="coerce").to_numpy(dtype=float)
item_prices = pd.to_numeric(items["item_price"], errors="coerce").to_numpy(dtype=float)
phone_hidden = items["item_is_phone_hidden"].fillna(False).to_numpy(dtype=float)
message_forbidden = items["item_is_message_forbidden"].fillna(False).to_numpy(dtype=float)
parameter_tokens = [set(text.split()) for text in items["clean_parameters"]]

feature_names = [
    "word_score", "char_score", "score_sum", "score_product",
    "same_location", "same_category", "category_specified",
    "query_length", "delivery_search", "filter_overlap",
    "rating", "rating_missing", "log_reviews", "reviews_missing",
    "log_price", "price_missing", "phone_hidden", "message_forbidden",
]


def make_features(row, record):
    """Рассчитывает 18 признаков для кандидатов одного запроса."""
    positions = record["positions"]
    count = len(positions)
    words = record["word_score"]
    chars = record["char_score"]
    same_location = item_locations[positions] == row["search_location_id"]
    category_specified = pd.notna(row["search_category"]) and row["search_category"] != 0
    same_category = (item_categories[positions] == row["search_category"]) & category_specified

    filter_words = set(clean_text(row["search_infm_params_text"]).split())
    overlap = []
    for position in positions:
        common = filter_words.intersection(parameter_tokens[position])
        overlap.append(len(common) / max(1, len(filter_words)))

    ratings = item_ratings[positions]
    reviews = item_reviews[positions]
    prices = item_prices[positions]
    delivery = row["search_is_delivery_search"]
    if pd.isna(delivery):
        delivery = 0

    # Собираем столбцы в порядке feature_names.
    features = np.column_stack([
        words, chars, words + chars, words * chars,
        same_location, same_category, np.full(count, category_specified),
        np.full(count, len(clean_text(row["search_query"]).split())),
        np.full(count, delivery), overlap,
        np.nan_to_num(ratings), np.isnan(ratings),
        np.log1p(np.maximum(np.nan_to_num(reviews), 0)), np.isnan(reviews),
        np.log1p(np.maximum(np.nan_to_num(prices), 0)), np.isnan(prices),
        phone_hidden[positions], message_forbidden[positions],
    ])
    return features.astype(np.float32)

## 13. Обучающие пары

Известный выбор получает метку 1, остальные кандидаты — условную метку 0. Сохраняются все найденные положительные и до 80 случайных отрицательных примеров на запрос. Суммарные веса классов внутри запроса равны. Группы без примеров обоих классов пропускаются при обучении, но остаются при оценке качества.

In [20]:
rng = np.random.default_rng(SEED)
feature_parts = []
label_parts = []
weight_parts = []
skipped_queries = 0

for number in range(len(train_part)):
    row = train_part.iloc[number]
    record = train_records[number]
    positions = record["positions"]
    # Метка 0 означает отсутствие записанного выбора, а не плохое объявление.
    labels = []
    for position in positions:
        was_selected = item_ids[position] in row["relevant_items"]
        labels.append(was_selected)
    labels = np.array(labels, dtype=bool)
    positive = np.flatnonzero(labels)
    negative = np.flatnonzero(~labels)

    if len(positive) == 0 or len(negative) == 0:
        skipped_queries += 1
        continue

    negative = rng.choice(negative, size=min(80, len(negative)), replace=False)
    selected = np.concatenate([positive, negative])
    features = make_features(row, record)
    weights = np.concatenate([
        np.full(len(positive), 0.5 / len(positive)),
        np.full(len(negative), 0.5 / len(negative)),
    ])

    feature_parts.append(features[selected])
    label_parts.append(labels[selected].astype(int))
    weight_parts.append(weights)

assert feature_parts, "Нет обучающих пар после поиска кандидатов"
X_train = np.concatenate(feature_parts)
y_train = np.concatenate(label_parts)
sample_weights = np.concatenate(weight_parts)
sample_weights = sample_weights / sample_weights.mean()

print("Размер обучающей таблицы:", X_train.shape)
print("Положительных примеров:", y_train.sum())
print("Пропущено запросов при обучении:", skipped_queries)
joblib.dump({"X_train": X_train, "y_train": y_train, "sample_weights": sample_weights},
            "outputs/v2/training_pairs.joblib")


Размер обучающей таблицы: (419760, 18)
Положительных примеров: 5360
Пропущено запросов при обучении: 820


['outputs/v2/training_pairs.joblib']

## 14. Градиентный бустинг

HistGradientBoostingClassifier: 120 итераций, до 15 листьев, learning_rate=0.08, min_samples_leaf=30, L2=1. Ранняя остановка отключена, чтобы избежать случайного внутреннего разбиения пар одного запроса. Оценка класса 1 используется для сортировки, не как откалиброванная вероятность выбора. Метод выбирается по dev Recall@50.

При равных dev-оценках предпочтение отдаётся принятой модели. Код не заменяет исходный answer.csv автоматически.

In [21]:
model = HistGradientBoostingClassifier(
    max_iter=120,
    max_leaf_nodes=15,
    learning_rate=0.08,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=SEED,
)

with threadpool_limits(limits=4):
    model.fit(X_train, y_train, sample_weight=sample_weights)


def model_predictions(table, records, predictor=None):
    """Выбирает до 50 кандидатов с помощью обученной модели model."""
    if predictor is None:
        predictor = model
    predictions = []
    with threadpool_limits(limits=4):
        for number in range(len(table)):
            row = table.iloc[number]
            record = records[number]
            positions = record["positions"]
            if len(positions) == 0:
                predictions.append(positions)
                continue
            features = make_features(row, record)
            # Берём оценки класса 1: известный выбор объявления.
            class_scores = predictor.predict_proba(features)
            scores = class_scores[:, 1]
            order = np.argsort(-scores, kind="stable")[:TOP_K]
            predictions.append(positions[order])
    return predictions


dev_model_predictions = model_predictions(dev_part, dev_records)
dev_model_score, _ = evaluate_lists(dev_part, dev_model_predictions)
results.append({"method": "gradient_boosting", "dev_recall_at_50": dev_model_score})

# Сравнение с принятой моделью на тех же dev-запросах.
previous_dev_predictions = model_predictions(dev_part, dev_records, previous_model)
previous_dev_score, _ = evaluate_lists(dev_part, previous_dev_predictions)
results.insert(0, {"method": "previous_model", "dev_recall_at_50": previous_dev_score})
print("Dev текущей модели:", previous_dev_score)
print("Dev модели на 6000 запросах:", dev_model_score)
comparison = pd.DataFrame(results).sort_values("dev_recall_at_50", ascending=False, kind="stable")
chosen_method = comparison.iloc[0]["method"]
comparison.to_csv("outputs/v2/dev_comparison.csv", index=False)
print("Выбранный на dev метод:", chosen_method)
comparison
joblib.dump({"model": model, "feature_names": feature_names}, "outputs/v2/model_6000.joblib")


Dev текущей модели: 0.788
Dev модели на 6000 запросах: 0.788
Выбранный на dev метод: previous_model


['outputs/v2/model_6000.joblib']

## 15. Новая контрольная выборка

Сравниваются принятая модель и метод, выбранный на dev. Все тексты старого test исключены. Дополнительно фиксируется результат новой модели независимо от выбора метода. После этой оценки настройки не меняются.

In [22]:
def selected_predictions(table, records):
    """Применяет метод, выбранный на dev, к запросам и их кандидатам."""
    if chosen_method == "previous_model":
        return model_predictions(table, records, previous_model)
    if chosen_method == "word_tfidf":
        return [record["baseline"] for record in records]
    if chosen_method == "gradient_boosting":
        return model_predictions(table, records)
    return hybrid_predictions(table, records, hybrid_weights[chosen_method])


test_records = retrieve_candidates(test_part)
test_baseline_predictions = model_predictions(test_part, test_records, previous_model)
test_selected_predictions = selected_predictions(test_part, test_records)

test_baseline, baseline_recalls = evaluate_lists(test_part, test_baseline_predictions)
test_selected, selected_recalls = evaluate_lists(test_part, test_selected_predictions)
test_ceiling, _ = evaluate_lists(test_part, [record["positions"] for record in test_records])

difference = selected_recalls - baseline_recalls
bootstrap_groups = pd.DataFrame({
    "text": test_part["clean_query"], "difference": difference,
}).groupby("text")["difference"].agg(["sum", "count"])
rng = np.random.default_rng(SEED)
bootstrap_differences = []

for _ in range(2000):
    sample = rng.integers(0, len(bootstrap_groups), len(bootstrap_groups))
    sampled_groups = bootstrap_groups.iloc[sample]
    bootstrap_differences.append(sampled_groups["sum"].sum() / sampled_groups["count"].sum())

interval = np.quantile(bootstrap_differences, [0.025, 0.975])
print("Recall@50 текущей модели на новом test:", round(test_baseline, 4))
print("Выбранный метод на test:", round(test_selected, 4))
print("Recall расширенного списка на test:", round(test_ceiling, 4))
print("Разница:", round(test_selected - test_baseline, 4))
print("Приблизительный 95% интервал разницы:", interval.round(4))
fresh_model_predictions = model_predictions(test_part, test_records, model)
fresh_model_score, _ = evaluate_lists(test_part, fresh_model_predictions)
print("Recall@50 модели на 6000 запросах на новом test:", fresh_model_score)


Обработано запросов: 320


Recall@50 текущей модели на новом test: 0.7763
Выбранный метод на test: 0.7763
Recall расширенного списка на test: 0.8433
Разница: 0.0
Приблизительный 95% интервал разницы: [0. 0.]


Recall@50 модели на 6000 запросах на новом test: 0.7813333333333333


## 16. Ошибки

Различаются ошибки поиска кандидатов и ошибки отбора: правильное объявление либо отсутствует в расширенном списке, либо не проходит из него в финальные 50. Примеры сохраняются для анализа; настройки после просмотра test не менялись.

In [23]:
error_rows = []

for number, row in test_part.iterrows():
    if selected_recalls[number] == 1:
        continue
    relevant = row["relevant_items"]
    retrieved = set(item_ids[test_records[number]["positions"]])
    selected = set(item_ids[test_selected_predictions[number]])
    missed = sorted(relevant - selected)
    titles = items_by_id.loc[missed, "item_title_raw"].tolist()
    error_rows.append({
        "query": row["search_query"],
        "location": row["search_location_id"],
        "recall_at_50": selected_recalls[number],
        "relevant_count": len(relevant),
        "relevant_in_candidates": len(relevant.intersection(retrieved)),
        "missed_titles": " | ".join(titles),
    })

errors = pd.DataFrame(error_rows)
errors.to_csv("outputs/v2/test_errors.csv", index=False)
errors.head(10)

,query,location,recall_at_50,relevant_count,relevant_in_candidates,missed_titles
0,купить диплом о высшем образовании,637640,0.0,1,0,"Снятие ии, Повышение Оригинальности, Рерайт"
1,работы в огороде пермский район селокояново,644200,0.0,1,0,"Расчистка участка, уборка территории"
2,замена переднего стекла айфона,637640,0.0,1,0,Ремонт телефонов iPhone и Android
3,ремонт акустических колонок,646600,0.0,1,0,Ремонт Электроники
4,аренда авто на дальние поездки,107621,0.0,1,0,Авто в аренду для поездки в отпуск
5,найти работника,642410,0.0,1,1,Помощь по хозяйству (калым )
6,бригада строителей чехов,637680,0.0,1,0,Ремонт и строительство в Чехове под ключ
7,ремонт стеклопакетов дверей,653240,0.0,1,0,Ремонт пластиковых окон и дверей
8,ремонт музыкальных колонок,637640,0.0,1,0,Ремонт динамиков и акустики
9,автовышка,107621,0.0,1,0,"Аренда и услуги телескопической автовышки, 300..."


## 17. Предсказания для бенчмарка

Используется метод, выбранный на dev, и та же модель, которая оценивалась на test. Дополнительного обучения на контрольной разметке нет.

In [24]:
benchmark_records = retrieve_candidates(queries)
benchmark_predictions = selected_predictions(queries, benchmark_records)

answer_rows = []
for positions in benchmark_predictions:
    answer_rows.append(" ".join(item_ids[positions]))

answer = pd.DataFrame({
    "query_id": queries["query_id"].tolist(),
    "answer": answer_rows,
})
answer.head(3)

Обработано запросов: 320


Обработано запросов: 640


Обработано запросов: 960


Обработано запросов: 1280


Обработано запросов: 1600


Обработано запросов: 1920


Обработано запросов: 2240


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 73ba3172bc67d2dd fa4ba2c2cf71...
1,JTrdTaZJvSiLPkXj,dfa239d031b11832 2f09017b0819844e 62a6515c16d0...
2,LZCZNoVG4AFUkVRJ,c5d21c0e8c1cdb2d d8fce513e4f000a7 10eb6d6da323...


## 18. Проверка answer.csv

Проверяются две колонки, полный набор query_id, отсутствие повторов, лимит 50, формат item_id и принадлежность корпусу. После записи CSV проверяется его чтение без изменения строк.

In [25]:
assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique
assert len(answer) == len(queries)
assert set(answer["query_id"]) == set(queries["query_id"])

for text in answer["answer"]:
    candidates = text.split()
    assert len(candidates) <= TOP_K
    assert len(candidates) == len(set(candidates))
    assert set(candidates).issubset(corpus_ids)
    assert all(re.fullmatch("[0-9a-f]{16}", item_id) for item_id in candidates)

answer.to_csv("answer_v2.csv", index=False, encoding="utf-8")
saved_answer = pd.read_csv("answer_v2.csv", dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(answer, saved_answer)

empty_answers = int((answer["answer"] == "").sum())
print("Файл сохранён. Запросов:", len(answer))
print("Пустых ответов:", empty_answers)

Файл сохранён. Запросов: 2452
Пустых ответов: 0


## 19. Сохранение результатов

Сохраняются модель, порядок признаков, метрики и время полного расчёта. Файл модели не заменяет текстовые индексы: полное воспроизведение выполняется из ноутбука.

In [26]:
import joblib

selected_model = previous_model if chosen_method == "previous_model" else model
joblib.dump({"model": selected_model, "feature_names": feature_names}, "outputs/v2/selector.joblib")

metrics = {
    "seed": SEED,
    "train_queries": len(train_part),
    "dev_queries": len(dev_part),
    "test_queries": len(test_part),
    "eligible_queries": len(eligible),
    "train_pairs": len(y_train),
    "skipped_train_queries": skipped_queries,
    "chosen_method": chosen_method,
    "dev_results": results,
    "dev_candidate_recall": dev_ceiling,
    "test_previous_model_recall_at_50": test_baseline,
    "test_model_6000_recall_at_50": fresh_model_score,
    "test_selected_recall_at_50": test_selected,
    "test_candidate_recall": test_ceiling,
    "test_difference_95_interval": interval.tolist(),
    "benchmark_queries": len(answer),
    "empty_answers": empty_answers,
    "elapsed_seconds": round(time.time() - start_time, 1),
}
with open("outputs/v2/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

print("По dev выбран:", chosen_method)
print("На новом test текущая модель:", round(test_baseline, 4))
print("На test выбранный метод:", round(test_selected, 4))
print("Изменение в процентных пунктах:", round(100 * (test_selected - test_baseline), 2))
print("Объявления вне расширенного списка ограничивают Recall уровнем:", round(test_ceiling, 4))
if interval[0] > 0:
    print("На этой выборке интервал разницы положительный: улучшение устойчиво к bootstrap-перевыборке.")
else:
    print("Интервал включает ноль или лежит ниже: уверенно утверждать улучшение по этой проверке нельзя.")
print("Время выполнения, секунд:", metrics["elapsed_seconds"])

По dev выбран: previous_model
На новом test текущая модель: 0.7763
На test выбранный метод: 0.7763
Изменение в процентных пунктах: 0.0
Объявления вне расширенного списка ограничивают Recall уровнем: 0.8433
Интервал включает ноль или лежит ниже: уверенно утверждать улучшение по этой проверке нельзя.
Время выполнения, секунд: 707.6


## Результаты эксперимента

Фактические метрики сохранены в выводах ячеек и outputs/v2/metrics.json. Метрики прежнего test напрямую с новой выборкой не сравниваются: обе модели оцениваются на одинаковых новых запросах.

## Ограничения

Локальная выборка ограничена пересечением train с корпусом. Невыбранные объявления не обязательно нерелевантны. В бенчмарке чаще встречается категория 0. Обучение ограничено 6000 группами, производственная задержка не измерялась. Дальнейшее развитие — расширение обучения и повышение полноты кандидатов с новой независимой проверкой.